In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta
import numpy as np

In [ ]:
# 신체 계측 정보 경로 
target_dir = '/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Weight-and-Height/de'

# 환자별 노이즈 정보
noise_df = pd.read_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
noise_df['new_pt_id'] = noise_df['new_pt_id'].astype(str)

pt_id_map = noise_df.set_index('pt_id')['new_pt_id'].to_dict()
date_noise_map = noise_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_noise_map = noise_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(target_dir, '**/*.csv'), recursive = True)
print(f"총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제
print("\n 컬럼 제거")

columns_to_drop = [
    'prone_bone', 'chest', 'head', 'sit_height'
]

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        total_cols_to_remove = len(columns_to_drop)
        cols_in_df = [col for col in columns_to_drop if col in df.columns]
        removed_cols_count = len(cols_in_df)
        
        if removed_cols_count > 0:
            df.drop(columns = cols_in_df, inplace = True)
            
        df.to_csv(file, index=False, encoding = 'utf-8')
        
        print(f"{file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")
    
    except Exception as e:
        print(f"[❗ 오류] {file} → {e}")

In [ ]:
# pt_id 치환
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        
        if 'pt_id' in df.columns:
            df['pt_id'] = df['pt_id'].map(pt_id_map).fillna(df['pt_id'])
        
        
        df.to_csv(file, index = False, encoding = 'utf-8')
        print(f"pt_id 변경 완료: {file}")
        
    except Exception as e:
        print(f"[pt_id 오류] {file} -> {e}")

In [ ]:
# hos_dt_first 처리
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        
        if 'hos_dt_first' in df.columns:
            df = df.rename(columns = {'hos_dt_first': 'host_dt_first'})
        
        if 'host_dt_first' in df.columns:
            df['host_dt_first'] = pd.to_datetime(df['host_dt_first'], errors = 'coerce')
            
            df['host_dt_first'] = df['host_dt_first'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D") \
                + pd.to_timedelta(df['pt_id'].map(time_noise_map).fillna(0).astype(int), unit = "m")
            
            print(df.head(3))
            
            df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"host_dt_first 완료: {file}")
            
    
    except Exception as e:
        print(f"[host_dt_first 오류] {file} -> {e}")

In [ ]:
# tx_date 처리
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype=str, sep=",", quotechar ='"', lineterminator = '\n')
        
        if 'tx_date' in df.columns:
            df['tx_date'] = pd.to_datetime(df['tx_date'], errors = 'coerce')
            pt_ids = df['pt_id']
            df['tx_date'] = df['tx_date'] + pt_ids.map(lambda x: timedelta(days = date_noise_map.get(x, 0)))
            
            print(df.head(3))

            df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"tx_date 완료: {file}")
    
    except Exception as e:
        print(f"[tx_date 오류] {file} -> {e}")

In [ ]:
# body_ms_dt 처리
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype=str, sep=",", quotechar ='"', lineterminator = '\n')
        
        if 'body_ms_dt' in df.columns:
            df['body_ms_dt'] = pd.to_datetime(df['body_ms_dt'], errors = 'coerce')
            
            df['body_ms_dt'] = df['body_ms_dt'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D") \
                + pd.to_timedelta(df['pt_id'].map(time_noise_map).fillna(0).astype(int), unit = "m")
           
            print(df.head(3))
            
            df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"body_ms_dt 완료: {file}")
    
    except Exception as e:
        print(f"[body_ms_dt 오류] {file} -> {e}")

In [ ]:
# vs_date 처리
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype=str, sep=",", quotechar ='"', lineterminator = '\n')

        if 'vs_date' in df.columns:
            df['vs_date'] = pd.to_datetime(df['vs_date'], errors = 'coerce')
            pt_ids = df['pt_id']
            df['vs_date'] = df['vs_date'] + pt_ids.map(lambda x: timedelta(days = date_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index = False, encoding = 'utf-8')
            print(f"vs_date 완료: {file}")
    
    except Exception as e:
        print(f"[vs_date 오류] {file} -> {e}")

In [ ]:
# weight 범주화
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        
        # weight 처리
        if 'weight' in df.columns:
            df['weight'] = pd.to_numeric(df['weight'], errors = 'coerce').fillna(0)

            df['weight'] = np.floor(df['weight'].astype(float) / 5) * 5
            
            df.to_csv(file, index = False, encoding = 'utf-8')
            print(df['weight'].head(5))
 
    
            print(f"범주화 완료: {file}")
        else:
            print(f"'weight' 컬럼 없음: {file}")
            
    except Exception as e:
        print(f"[weight 오류] {file} -> {e}")

In [ ]:
# height 범주화
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        
        # weight 처리
        if 'height' in df.columns:
            df['height'] = np.floor(df['height'].astype(float) / 5) * 5
            
            df.to_csv(file, index = False, encoding = 'utf-8')
            print(df['height'].head(5))
 
    
            print(f"범주화 완료: {file}")
        else:
            print(f"'height' 컬럼 없음: {file}")
            
    except Exception as e:
        print(f"[height 오류] {file} -> {e}")